# Midterm Practice Notebook 3: The Complete Flow

This practice notebook simulates the complete pipeline: from raw data generation to Scikit-learn preprocessing (Lecture 7) and connecting it directly into PyTorch (Lecture 8).

**How to use this notebook:**
1. Write your solution in the designated code blocks.
2. Run the cell.
3. If the `assert` statements pass without throwing an error, your code is correct!

## Task 1: Generate a Synthetic Tabular Dataset

Let's create a DataFrame with numeric features, categorical features, and a target variable.

1. Use `np.linspace` to create an array `income` with 100 values from 30000 to 120000.
2. Create a categorical list `cities` that repeats `['Khobar', 'Dammam', 'Jubail', 'Dhahran']` 25 times so it has 100 elements.
3. Create a target array `approved` containing 100 random binary labels (0 or 1). Use `np.random.randint(0, 2, 100)`.
4. Combine them into a Pandas DataFrame `df` with columns `['income', 'city', 'approved']`.

In [4]:
import numpy as np
import pandas as pd

np.random.seed(42)

# TODO: Generate the data and create the DataFrame
income = np.linspace(30_000, 120_000, num=100)
cities = ["Khobar", "Dammam", "Jubail", "Dhahran"] * 25
approved = np.random.randint(0, 2, 100)
df = pd.DataFrame(data={
    "income": income,
    "city": cities,
    "approved": approved
})



# --- ASSERTION BLOCK ---
assert 'df' in locals(), "DataFrame 'df' not defined."
assert df.shape == (100, 3), "DataFrame should have 100 rows and 3 columns."
assert list(df.columns) == ['income', 'city', 'approved'], "Column names do not match."
assert df['income'].iloc[0] == 30000, "Income should start at 30000."
print("Task 1: Passed! ✓")

Task 1: Passed! ✓


## Task 2: Train/Test Split (The Correct Workflow)

According to Lecture 7, we must *never* fit a scaler on test data (Workflow B is correct).

1. Separate features `X` (income and city) and target `y` (approved).
2. Split them into `X_train`, `X_test`, `y_train`, `y_test` using `train_test_split` with `test_size=0.2` and `random_state=42`.

In [6]:
from sklearn.model_selection import train_test_split

# TODO: Split the data
X = df[["income", "city"]]
y = df["approved"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)



# --- ASSERTION BLOCK ---
assert 'X_train' in locals() and 'X_test' in locals(), "Split variables not defined."
assert X_train.shape == (80, 2), "X_train should have 80 rows and 2 columns."
assert X_test.shape == (20, 2), "X_test should have 20 rows and 2 columns."
print("Task 2: Passed! ✓")

Task 2: Passed! ✓


## Task 3: The Scikit-learn Pipeline

Now, build a `ColumnTransformer` to handle the different feature types.

1. Use `StandardScaler` on the `'income'` column.
2. Use `OneHotEncoder(sparse_output=False)` on the `'city'` column.
3. Fit the transformer on `X_train`, and then transform **both** `X_train` and `X_test`.
   *(Store the results as `X_train_processed` and `X_test_processed`)*

In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# TODO: Build the ColumnTransformer and apply it correctly
preprocessor = ColumnTransformer([
    ("scaler", StandardScaler(), ["income"]),
    ("encoder", OneHotEncoder(sparse_output=False), ["city"]),
])
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed =  preprocessor.transform(X_test)



# --- ASSERTION BLOCK ---
assert 'X_train_processed' in locals() and 'X_test_processed' in locals(), "Processed variables not defined."
assert isinstance(X_train_processed, np.ndarray), "Output must be a NumPy array."
assert X_train_processed.shape == (80, 5), "Shape should be (80, 5) -> 1 scaled income + 4 city categories."
assert X_test_processed.shape == (20, 5), "Test set should have shape (20, 5)."
print("Task 3: Passed! ✓")

Task 3: Passed! ✓


## Task 4: PyTorch Custom Dataset

Now we move from the Scikit-learn world to the PyTorch world (Lecture 8).

Build a custom PyTorch Dataset class named `ApplicantDataset` that inherits from `Dataset` and implements `__init__`, `__len__`, and `__getitem__`.

Then, instantiate two datasets: `train_dataset` (using `X_train_processed` and `y_train`) and `test_dataset` (using `X_test_processed` and `y_test`). Ensure the features are `torch.float32` and labels are `torch.int64`.

In [16]:
import torch
from torch.utils.data import Dataset

# TODO: Create the Dataset class and instantiate train/test datasets
class ApplicantDataset(Dataset):
    def __init__(self, X, y):

        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y.values, dtype=torch.int64)

    def __len__(self):
         return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_dataset = ApplicantDataset(X_train_processed, y_train)
test_dataset = ApplicantDataset(X_test_processed, y_test)



# --- ASSERTION BLOCK ---
assert 'ApplicantDataset' in locals(), "ApplicantDataset class not defined."
assert 'train_dataset' in locals() and 'test_dataset' in locals(), "Dataset instances not defined."
assert len(train_dataset) == 80, "train_dataset length should be 80."
assert len(test_dataset) == 20, "test_dataset length should be 20."

features, label = train_dataset[0]
assert features.dtype == torch.float32, "Features must be float32."
assert label.dtype == torch.int64, "Labels must be int64."
print("Task 4: Passed! ✓")

Task 4: Passed! ✓


## Task 5: PyTorch DataLoader

The final step in the pipeline: "DataLoader handles batching, shuffling, and iterating through the dataset" (Lecture 8).

1. Create a `DataLoader` named `train_loader` using `train_dataset`, with a batch size of 16 and shuffling enabled.
2. Create a `DataLoader` named `test_loader` using `test_dataset`, with a batch size of 16 and shuffling **disabled** (we don't shuffle test sets!).

In [17]:
from torch.utils.data import DataLoader

# TODO: Create train_loader and test_loader
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)



# --- ASSERTION BLOCK ---
assert 'train_loader' in locals() and 'test_loader' in locals(), "DataLoaders not defined."
assert train_loader.batch_size == 16, "Train batch size must be 16."
assert test_loader.batch_size == 16, "Test batch size must be 16."

train_batch_x, train_batch_y = next(iter(train_loader))
assert train_batch_x.shape == (16, 5), "Train batch X shape should be (16, 5)."
print("Task 5: Passed! ✓ You have successfully connected Scikit-learn to PyTorch!")

Task 5: Passed! ✓ You have successfully connected Scikit-learn to PyTorch!
